# NERC

**TODO: Add description**

In [ ]:
# add all imports here
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
import os

from prettytable import PrettyTable

## Preprocessing
Change all instances of "geo" to "loc" and "gpe" to "misc" to match with the test set. 
Commented out since the new training set is already saved as a file to save time in future runs. 

In [ ]:
#start_time = time.time()
#unprocessed_data = pd.read_csv(filepath_or_buffer="data/train/ner_dataset.csv", sep=",", error_bad_lines=False, encoding= "unicode_escape")
#unprocessed_data = unprocessed_data[:10000]
#for i, tag in enumerate(unprocessed_data["Tag"]):
#    if "geo" in tag:
#        unprocessed_data["Tag"][i] = unprocessed_data["Tag"][i].replace("geo", "loc")
#    elif "gpe" in tag:
#        unprocessed_data["Tag"][i] = unprocessed_data["Tag"][i].replace("gpe", "misc")
#print(unprocessed_data["Tag"][102])
#print("--- %s seconds ---" % (time.time() - start_time))
#unprocessed_data.to_csv(path_or_buf="data/train/new_ner_dataset.csv", sep=",", index=False)

## EDA

In [ ]:
df = pd.read_csv('data/train/new_ner_dataset.csv', sep=",", error_bad_lines=False, encoding= "unicode_escape")

In [ ]:
df

In [ ]:
# first insight into the data
print("Total number of sentences: ", len(df["Sentence #"].unique()))

print("Total number of words: ", len(df["Word"]))
print("Total number of unique words: ", len(df["Word"].unique()))

print("Total number of tags: ", len(df["Tag"]))
print("Total number of unique tags: ", len(df["Tag"].unique()))



In [ ]:
# more insight into the data
print(df["Tag"].value_counts())

# plot the distribution of the tags
fig, ax = plt.subplots(figsize=(20,10))
sns.countplot(x="Tag", data=df)
# add counts and percentages to the plot
for p in ax.patches:
    ax.annotate('{:.3f}%'.format(100*p.get_height()/len(df)), (p.get_x()+0.1, p.get_height()+300))

# set the title of the plot
plt.title("Distribution of tags")
plt.show()
plt.savefig("figures/distribution_of_tags_barplot.png")





In [ ]:
# more insight into the data for POS tags (might be useful for the future)
print(df["POS"].value_counts())

# plot the distribution of the tags
fig, ax = plt.subplots(figsize=(20,10))
sns.countplot(x="POS", data=df)
# add counts and percentages to the plot
for p in ax.patches:
    ax.annotate('{:.3f}%'.format(100*p.get_height()/len(df)), (p.get_x()+0.1, p.get_height()+300))

# set the title of the plot
plt.title("Distribution of POS")
plt.show()
plt.savefig("figures/distribution_of_pos_barplot.png")



In [ ]:
# fill the sentence # column with the sentence number
df["Sentence #"] = df["Sentence #"].fillna(method="ffill")

In [ ]:

# compare the unique tags on the sentence level

# get the unique tags for each sentence
unique_tags = df.groupby("Sentence #")["Tag"].unique()
# find the number of unique tags for each sentence
unique_tags_count = unique_tags.apply(lambda x: len(x))
# print the number of unique tags for each sentence
print(unique_tags_count.value_counts())

# create a df with the number of unique tags for each sentence
d = pd.DataFrame(unique_tags_count)
d = d.reset_index()
d.columns = ["Sentence #", "Number of unique tags"]


#plot the distribution of the number of unique tags for each sentence
fig, ax = plt.subplots(figsize=(20,10))
sns.countplot(x="Number of unique tags", data=d)
# add percentages to the plot
for p in ax.patches:
    ax.annotate('{:.3f}%'.format(100*p.get_height()/len(d)), (p.get_x()+0.1, p.get_height()+300))


# set the title of the plot
plt.title("Distribution of number of unique tags per sentence")
plt.show()
plt.savefig("figures/distribution_of_unique_tags_per_sentence_barplot.png")




## MODEL TRAINING

Prepare the data

In [ ]:
import pandas
path = r'data/train/new_ner_dataset.csv'
kaggle_dataset = pandas.read_csv(path, error_bad_lines=False)
len(kaggle_dataset)

In [ ]:
test_data = pd.read_csv('data/test/NER-final-test.tsv', sep='\t')

Create train and test sets

In [ ]:
df_train = kaggle_dataset[:1050000]
df_test = test_data
print(len(df_train), len(df_test))

In [ ]:
df_train

In [ ]:
df_test

Training the model

In [ ]:
training_features = []
training_gold_labels = []
test_features = []
test_gold_labels = []

for index, row in df_train.iterrows():
    a_dict = {"words": row['Word']}
    training_features.append(a_dict)
    training_gold_labels.append(row['Tag'])

for index, row in df_test.iterrows():
    a_dict = {"words": row['token']}
    test_features.append(a_dict)
    test_gold_labels.append(row['BIO NER tag'])

# number of instances of train and test set
print("Number of training instances: ", len(training_features))
print("Number of test instances: ", len(test_features))

In [ ]:
from sklearn.feature_extraction import DictVectorizer
vec = DictVectorizer()
the_array = vec.fit_transform(training_features + test_features)
train_array = the_array[:len(training_features)]
test_array = the_array[len(training_features):]

In [ ]:
from sklearn import svm
lin_clf = svm.LinearSVC()
lin_clf.fit(train_array, training_gold_labels)

## EVALUATION

In [ ]:
predicted_labels = lin_clf.predict(test_array)
predicted_labels_real=[]
for i in predicted_labels:
    if i == "B-org":
         predicted_labels_real.append("B-ORG")
    elif i == "I-org":
         predicted_labels_real.append("I-ORG")
    elif i == "B-loc":
         predicted_labels_real.append("B-LOC")
    elif i == "I-loc":
         predicted_labels_real.append("I-LOC")
    elif i == "B-misc":
         predicted_labels_real.append("B-MISC")
    elif i == "B-per":
         predicted_labels_real.append("B-PER")
    elif i == "I-per":
         predicted_labels_real.append("I-PER")
    elif i == "I-art":
         predicted_labels_real.append("I-ART")
    elif i == "I-tim":
         predicted_labels_real.append("I-TIM")
    else:
        predicted_labels_real.append(i)

In [ ]:

from sklearn.metrics import accuracy_score
print("Accuracy: ", accuracy_score(test_gold_labels, predicted_labels_real))

from sklearn.metrics import confusion_matrix
#print("Confusion matrix: \n", confusion_matrix(test_gold_labels, predicted_labels))

from sklearn.metrics import classification_report
print("Classification report: \n", classification_report(test_gold_labels, predicted_labels_real))

In [ ]:
print(predicted_labels_real)
print(test_gold_labels)

### VALIDATION SET

### TEST SET